# Покрытие тарифов: Excel vs Datamart

Тетрадка делает проверки:
1. Покрытие тарифом для `agr_id` из Excel
2. Покрытие тарифом для `agr_id` из Datamart за месяц
3. `agr_id`, которых вообще нет в `ods.scd1_z_r2_ip_merchants`
4. Примеры строк без тарифа

In [ ]:
import pandas as pd

# Параметры
excel_path = "/home/jovyan/documents/Equaring/Проверки/02_февраль.xlsx"  # путь к Excel
agr_col = "agr_id"                      # имя колонки agr_id в Excel
month_start = "2026-02-01"              # YYYY-MM-01

In [ ]:
# 1) agr_id из Excel
df = pd.read_excel(excel_path)

excel_agr_df = pd.DataFrame({
    "agr_id_str": (
        df[agr_col]
        .astype(str)
        .str.strip()
        .replace({"": None, "nan": None, "None": None})
        .dropna()
        .str.replace(r"\\.0$", "", regex=True)
        .drop_duplicates()
    )
})

excel_agr_df["agr_id_num"] = pd.to_numeric(excel_agr_df["agr_id_str"], errors="coerce")
excel_agr_df = excel_agr_df.dropna(subset=["agr_id_num"]).copy()
excel_agr_df["agr_id_num"] = excel_agr_df["agr_id_num"].astype("int64")
excel_agr_df = excel_agr_df[["agr_id_num"]].drop_duplicates()

print("excel_rows_total =", len(df))
print("excel_unique_agr =", len(excel_agr_df))
excel_agr_df.head()

In [ ]:
# 2) agr_id из Datamart за месяц
with imp:
    dm = imp.fetch(f"""
        select distinct cast(agr_id as string) as agr_id_str
        from sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart
        where trx_month = cast('{month_start}' as date)
          and agr_id is not null
    """)

dm_agr_df = dm.copy()
dm_agr_df["agr_id_str"] = dm_agr_df["agr_id_str"].astype(str).str.strip().str.replace(r"\\.0$", "", regex=True)
dm_agr_df["agr_id_num"] = pd.to_numeric(dm_agr_df["agr_id_str"], errors="coerce")
dm_agr_df = dm_agr_df.dropna(subset=["agr_id_num"]).copy()
dm_agr_df["agr_id_num"] = dm_agr_df["agr_id_num"].astype("int64")
dm_agr_df = dm_agr_df[["agr_id_num"]].drop_duplicates()

print("datamart_unique_agr =", len(dm_agr_df))
dm_agr_df.head()

In [ ]:
# 3) Карта тарифов по agr_id из R2
with imp:
    tariff_map = imp.fetch("""
        select
            cast(m.id as bigint) as agr_id_num,
            m.c_tariff_plan as tariff_plan_id,
            tp.c_name as tariff_plan_name
        from ods.scd1_z_r2_ip_merchants m
        left join ods.scd1_z_r2_tariff_plan tp
          on tp.id = m.c_tariff_plan
    """)

# Заполненным считаем тариф, который не null и не 0/пусто
tariff_map["tariff_plan_id_str"] = tariff_map["tariff_plan_id"].astype(str).str.strip()
tariff_map["tariff_is_filled"] = (
    tariff_map["tariff_plan_id"].notna()
    & (~tariff_map["tariff_plan_id_str"].isin(["", "0", "0.0", "0.00", "nan", "None"]))
)

# Если по agr_id несколько строк, считаем заполненным, если хотя бы одна строка заполнена
tariff_by_agr = (
    tariff_map.groupby("agr_id_num", as_index=False)["tariff_is_filled"]
    .max()
)

# Отдельно список agr_id, которые в принципе есть в R2
r2_agr_set = set(tariff_map["agr_id_num"].dropna().astype("int64").unique())

print("r2_unique_agr =", len(r2_agr_set))

In [ ]:
# 4) Покрытие тарифа: Excel
excel_check = excel_agr_df.merge(tariff_by_agr, on="agr_id_num", how="left")
excel_check["tariff_is_filled"] = excel_check["tariff_is_filled"].fillna(False)

excel_total = len(excel_check)
excel_filled = int(excel_check["tariff_is_filled"].sum())
excel_empty = int(excel_total - excel_filled)
excel_rate = round(100.0 * excel_filled / excel_total, 2) if excel_total else 0.0

excel_not_in_r2 = excel_check.loc[~excel_check["agr_id_num"].isin(r2_agr_set), ["agr_id_num"]].copy()
excel_missing_tariff = excel_check.loc[(excel_check["agr_id_num"].isin(r2_agr_set)) & (~excel_check["tariff_is_filled"]), ["agr_id_num"]].copy()

pd.DataFrame([
    {"scope": "excel", "metric": "total_agr", "value": excel_total},
    {"scope": "excel", "metric": "tariff_filled", "value": excel_filled},
    {"scope": "excel", "metric": "tariff_empty", "value": excel_empty},
    {"scope": "excel", "metric": "tariff_fill_rate_pct", "value": excel_rate},
    {"scope": "excel", "metric": "agr_not_in_r2", "value": len(excel_not_in_r2)},
    {"scope": "excel", "metric": "agr_in_r2_but_tariff_empty", "value": len(excel_missing_tariff)},
])

In [ ]:
# 5) Покрытие тарифа: Datamart
dm_check = dm_agr_df.merge(tariff_by_agr, on="agr_id_num", how="left")
dm_check["tariff_is_filled"] = dm_check["tariff_is_filled"].fillna(False)

dm_total = len(dm_check)
dm_filled = int(dm_check["tariff_is_filled"].sum())
dm_empty = int(dm_total - dm_filled)
dm_rate = round(100.0 * dm_filled / dm_total, 2) if dm_total else 0.0

dm_not_in_r2 = dm_check.loc[~dm_check["agr_id_num"].isin(r2_agr_set), ["agr_id_num"]].copy()
dm_missing_tariff = dm_check.loc[(dm_check["agr_id_num"].isin(r2_agr_set)) & (~dm_check["tariff_is_filled"]), ["agr_id_num"]].copy()

pd.DataFrame([
    {"scope": "datamart", "metric": "total_agr", "value": dm_total},
    {"scope": "datamart", "metric": "tariff_filled", "value": dm_filled},
    {"scope": "datamart", "metric": "tariff_empty", "value": dm_empty},
    {"scope": "datamart", "metric": "tariff_fill_rate_pct", "value": dm_rate},
    {"scope": "datamart", "metric": "agr_not_in_r2", "value": len(dm_not_in_r2)},
    {"scope": "datamart", "metric": "agr_in_r2_but_tariff_empty", "value": len(dm_missing_tariff)},
])

In [ ]:
# 6) Примеры проблемных agr_id
print("Excel: agr_id not in R2 (top 50)")
display(excel_not_in_r2.head(50))

print("Excel: agr_id in R2 but tariff empty (top 50)")
display(excel_missing_tariff.head(50))

print("Datamart: agr_id not in R2 (top 50)")
display(dm_not_in_r2.head(50))

print("Datamart: agr_id in R2 but tariff empty (top 50)")
display(dm_missing_tariff.head(50))